In [1]:
import os
import sys

import jsonlines
import pandas as pd

sys.path.append(f"{os.environ['MDS']}/code")
from project_utils import load_config


In [2]:
results_path = f"{os.environ['MDS']}/results"

experiments = load_config(f"{os.environ['MDS']}/code/experiments.json")


In [8]:
df = []
for model in experiments["models"]:
    if model == "ground_truth":
        continue
    for dataset in experiments["datasets"]:
        for split in experiments["splits"]:
            file_path = f"{results_path}/{model}/{dataset}/{split}/core_output.jsonl"
            if not os.path.exists(file_path):
                continue
            
            with jsonlines.open(file_path) as reader:
                for entry in reader.iter():
                    summary_claims = entry.get("meta", {}).get("claims", [])
                    if summary_claims:
                        summary_claims = [c['text'] for c in summary_claims if c['score']==1]
                    row = {
                        "id": entry["id"],
                        "claims": summary_claims,
                        "model": model,
                        "dataset": dataset,
                        "split": split
                    }
                    df.append(row)
                    break

df = pd.DataFrame(df)

In [9]:
df

,id,claims,model,dataset,split
0,https://www.reddit.com/r/arma/comments/1fy1xp/...,"[The explanation is about thermal imaging., Th...",allenai/PRIMERA-multinews,convosumm/reddit,val
1,https://www.reddit.com/r/appletv/comments/43s1...,[The app is reportedly experiencing some issue...,allenai/PRIMERA-multinews,convosumm/reddit,test
2,cmp-lg9807007,[Energy function optimization is used in very ...,allenai/PRIMERA-multinews,multixscience,val
3,cmp-lg9709004,[The method is evaluated over the Brown Corpus...,allenai/PRIMERA-multinews,multixscience,test
4,47100,[Ki Suk Han was pushed onto the tracks of a Ne...,allenai/PRIMERA-multinews,multinews_plus,val
5,56000,"[Trump expressed optimism about the summit., O...",allenai/PRIMERA-multinews,multinews_plus,test
6,https://www.reddit.com/r/arma/comments/1fy1xp/...,[],allenai/PRIMERA-multixscience,convosumm/reddit,val
7,https://www.reddit.com/r/appletv/comments/43s1...,[The problem with the CNNgo app has existed fo...,allenai/PRIMERA-multixscience,convosumm/reddit,test
8,cmp-lg9807007,[Energy function optimization is used in very ...,allenai/PRIMERA-multixscience,multixscience,val
9,cmp-lg9709004,[The method is evaluated over the Brown Corpus...,allenai/PRIMERA-multixscience,multixscience,test


In [19]:
for i, c in enumerate(claims):
    if c.get("score", -1) == 0:
        print(f"""{c['id']} {c['score']:.2}
        {c['decoded']}
        {c['meta']}
-----------------
""")
    if i > 10:
        break